```bash
CUDA_VISIBLE_DEVICES=0 vllm serve Qwen/Qwen2.5-7B-Instruct \
    --host 0.0.0.0 \
    --port 8084 \
    --gpu-memory-utilization 0.85 \
    --enable-prefix-caching \
    --dtype bfloat16 \
    --max_model_len 8000 \
    --trust-remote-code
```

```bash
CUDA_VISIBLE_DEVICES=1 vllm serve Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B \
    --host 0.0.0.0 \
    --port 8082 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

```bash
CUDA_VISIBLE_DEVICES=1 vllm serve Qwen/Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8083 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [1]:
from openai import OpenAI

OPENAI_API_KEY = "EMPTY"
OPENAI_API_BASE = "http://localhost:{PORT}/v1"

causal_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8084),
)
causal_model = causal_client.models.list().data[0].id

# skywork_prm_client = OpenAI(
#     api_key=OPENAI_API_KEY,
#     base_url=OPENAI_API_BASE.format(PORT=8082),
# )
# skywork_prm_model = skywork_prm_client.models.list().data[0].id

qwen_prm_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE.format(PORT=8083),
)
qwen_prm_model = qwen_prm_client.models.list().data[0].id


In [2]:
import re
import pandas as pd
from tqdm.auto import tqdm
from functools import partial
from transformers import AutoTokenizer
from multiprocessing import Pool, cpu_count
from constants.prompts_constants import (
    VERBOSE_TASK, CONSISE_TASK, EQ_TO_TEXT_TASK, CHANGE_NUMBERS_TASK
)

from utils.prompt_utils import get_augmentation_prompt, get_equivalence_prompt
from utils.io_utils import prepare_input, derive_step_rewards_vllm, prepare_batch_input_for_model

In [3]:
df = pd.read_parquet("data/processbench.parquet")

df.keys()

Index(['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label',
       'split', 'steps_len', 'per_step_len', 'Qwen2.5-Math-PRM-7B',
       'Skywork-o1-Open-PRM-Qwen-2.5-7B'],
      dtype='object')

In [16]:
def augmentor(index_row, task_text, client, model):
    _, row = index_row

    question, steps = row["problem"], row["steps"]
    prompt          = get_augmentation_prompt(question, steps, task_text)

    # call OpenAI
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],

    )
    content = resp.choices[0].message.content

    # grab the <response>…</response> block
    m = re.search(r"<response>(.*?)</response>", content, re.DOTALL)
    if not m:
        return {"aug_question":"", "aug_steps":[]}

    body = m.group(1).strip()

    # extract question
    q_m = re.search(r"<question>(.*?)</question>", body, re.DOTALL)
    aug_question = q_m.group(1).strip() if q_m else ""

    # extract steps: find all <step#>…</step#>
    aug_steps = re.findall(r"<step\d+>(.*?)</step\d+>", body, re.DOTALL)
    aug_steps = [s.strip() for s in aug_steps]

    return {"aug_question": aug_question, "aug_steps": aug_steps}

def equivalence_check(pair, client, model):
    (_, rowA), (_, rowB) = pair
    qA, stepsA = rowA["problem"], rowA["steps"]
    qB, stepsB = rowB["aug_question"], rowB["aug_steps"]

    prompt = get_equivalence_prompt(qA, stepsA, qB, stepsB)
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    content = resp.choices[0].message.content

    # extract the <response>…</response> block
    m = re.search(r"<response>(.*?)</response>", content, re.DOTALL)
    if not m: return False

    body = m.group(1)

    # grab question flag
    q_m = re.search(r"<question>\s*([YN])\s*</question>", body)
    question_flag = q_m.group(1) if q_m else "N"

    # grab all step flags into a list
    step_flags = re.findall(r"<step\d+>\s*([YN])\s*</step\d+>", body)

    # final check: question + every step must be "Y"
    all_flags = [question_flag] + step_flags
    return all(f == "Y" for f in all_flags)

def prm_scorer(questions, steps, client, model, batch_size=32):
    num_samples = len(questions)
    tokenizer = AutoTokenizer.from_pretrained(model)

    input_ids_all = []
    token_mask_all = []
    all_rewards = []

    for (question, step) in tqdm(zip(questions, steps), total=len(questions), desc="[PRM] Preparing input"):
        input_ids, token_mask = prepare_input(
                                model, 
                                problem=question, 
                                steps=step, 
                                tokenizer=tokenizer,
                                convert_to_list=True
        )
        input_ids_all.append(input_ids)
        token_mask_all.append(token_mask)


    for start_idx in tqdm(range(0, num_samples, batch_size), desc="[PRM] Scoring"):
        end_idx = start_idx + batch_size
    
        batch_input_ids = input_ids_all[start_idx:end_idx]
        batch_token_masks = token_mask_all[start_idx:end_idx]
    
        batch_input_ids, batch_token_masks = prepare_batch_input_for_model(batch_input_ids, batch_token_masks, pad_token_id=0)
    
        batch_logits = client.embeddings.create(
            input=batch_input_ids.cpu().tolist(),
            model=model,
        )
    
        rewards = derive_step_rewards_vllm(
            model,
            batch_logits,
            batch_token_masks,
            tokenizer
        )
    
        all_rewards.extend(rewards)
    return all_rewards

In [30]:
def attack(df_sample, 
    task_text, 
    prm_client, 
    experiment_name, 
    causal_client=causal_client
):
    causal_model = causal_client.models.list().data[0].id
    prm_model = prm_client.models.list().data[0].id

    # Apply the augmentor function to each row in the DataFrame
    aug_results = []
    for index_row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Augmenting"):
        aug_results.append(augmentor(index_row, task_text, client=causal_client, model=causal_model))
    df_aug = pd.DataFrame(aug_results)

    # Check equivalence
    equivalence_results = []
    for pair in tqdm(zip(df_sample.iterrows(), df_aug.iterrows()), total=len(df_sample), desc="Checking equivalence"):
        equivalence_results.append(equivalence_check(pair, client=causal_client, model=causal_model))
    df_aug["equivalence"] = equivalence_results

    # PRM Scorer
    rewards = prm_scorer(questions=df_aug["aug_question"].tolist(), 
                        steps=df_aug["aug_steps"].tolist(), 
                        client=prm_client, model=prm_model)
    df_aug[f"{prm_model}--rewards"] = rewards
    
    # concat the original and augmented DataFrames
    for key in df_aug.keys():
        df_sample[key] = df_aug[key]
    
    # Save the DataFrame to a CSV file
    df_sample.to_parquet(f"experiments/{experiment_name}.parquet", index=False)
    return df_sample

In [31]:
sample_size = 5
df_sample = df.sample(sample_size).reset_index(drop=True)

attack(df_sample, 
    task_text=VERBOSE_TASK, 
    prm_client=qwen_prm_client, 
    experiment_name="verbose_task_5"
)

Augmenting:   0%|          | 0/5 [00:00<?, ?it/s]

Checking equivalence:   0%|          | 0/5 [00:00<?, ?it/s]

[PRM] Preparing input:   0%|          | 0/5 [00:00<?, ?it/s]

[PRM] Scoring:   0%|          | 0/1 [00:00<?, ?it/s]

,id,generator,problem,steps,final_answer_correct,label,split,steps_len,per_step_len,Qwen2.5-Math-PRM-7B,Skywork-o1-Open-PRM-Qwen-2.5-7B,aug_question,aug_steps,equivalence,Qwen/Qwen2.5-Math-PRM-7B--rewards
0,olympiadbench-1,Qwen2.5-Math-72B-Instruct,Let $T=$ 4. Pyramid $L E O J S$ is a right squ...,"[To find the area of triangle \( [LEO] \), we ...",False,3,olympiadbench,5,"[330, 417, 378, 695, 66]","[0.98046875, 0.99609375, 0.98828125, 0.0571289...","[0.3191213844321178, 0.5664982418472789, 0.399...","We have a right square pyramid named LEOJS, wh...","[To begin, we need to recognize that the base ...",True,"[0.96484375, 0.953125, 0.80859375, 0.076171875]"
1,gsm8k-53,Qwen2-7B-Instruct,My wife wants to evenly split the check but wa...,"[Firstly, let's calculate the total bill inclu...",False,3,gsm8k,5,"[171, 108, 177, 369, 203]","[0.99609375, 1.0, 0.99609375, 0.23046875, 0.21...","[0.6173973000187004, 0.4215518210658887, 0.685...",My wife has expressed a desire to split the di...,[We start by calculating the additional amount...,True,"[0.8203125, 0.99609375, 0.2353515625, 0.033203..."
2,omnimath-78,Llama-3.1-70B-Instruct,Contessa is taking a random lattice walk in th...,"[To solve this problem, we need to consider th...",False,0,omnimath,12,"[388, 250, 260, 189, 302, 608, 151, 92, 164, 4...","[0.5, 0.1318359375, 0.65234375, 0.92578125, 0....","[0.1732882059293266, 0.16667540468797667, 0.16...",Contessa initiates a random walk on a 2D latti...,[To determine the probability that Contessa wi...,True,"[0.94921875, 0.365234375, 0.267578125, 0.58203..."
3,omnimath-360,Qwen2.5-72B-Instruct,Yannick has a bicycle lock with a 4-digit pass...,[To solve the problem of finding the maximum p...,False,6,omnimath,10,"[201, 198, 38, 157, 259, 270, 324, 202, 106, 138]","[1.0, 0.99609375, 1.0, 0.9921875, 0.890625, 0....","[0.020332353342658753, 0.02228618553922549, 0....","Yannick's bicycle lock has a 4-digit passcode,...","[To understand the problem, let's consider the...",True,"[0.96875, 0.58203125, 0.16015625, 0.1181640625..."
4,olympiadbench-392,Qwen2.5-Math-72B-Instruct,Triangle $A B C$ is inscribed in circle $\omeg...,"[To solve for the area of triangle \(PBC\), we...",False,1,olympiadbench,5,"[701, 610, 507, 456, 662]","[0.1904296875, 0.04248046875, 0.08203125, 0.58...","[0.021615332762647654, 0.028870906946287904, 0...",Triangle \(ABC\) is inscribed in a circle \(\o...,"[Initially, we identify point \(T\), which is ...",False,"[0.328125, 0.0079345703125, 0.2041015625, 0.59..."
